In [1]:
# ====================== ONE CELL COMPLETE CODE ======================

!pip install transformers datasets accelerate -q

import torch, math
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset
from transformers import set_seed

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# Text generation function
def generate_text(prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Baseline
prompts = [
    "This product is",
    "I bought this phone and",
    "The quality of this item"
]

print("\n=== BEFORE FINE-TUNING ===\n")
baseline = {}
for p in prompts:
    text = generate_text(p)
    baseline[p] = text
    print(f"Prompt: {p}\nOutput: {text}\n")

# Dataset
corpus = [
    "this phone has an amazing battery life and the camera quality is outstanding for the price.",
    "i bought this laptop for college and it handles all my assignments and coding projects perfectly.",
    "the sound quality of these headphones is incredible with deep bass and clear vocals.",
    "this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.",
    "great wireless earbuds with noise cancellation that blocks out all background sound.",
    "the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.",
    "this portable charger saved me during travel and it charges my phone three times on a single charge.",
    "the tablet screen is bright and colorful which makes watching movies a great experience.",
    "i love this fitness tracker because it motivates me to reach my daily exercise goals.",
    "this bluetooth speaker is compact but delivers surprisingly loud and clear audio."
]

dataset = Dataset.from_dict({"text": corpus})

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
split = tokenized.train_test_split(test_size=0.2)

# Training setup
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./gpt2-reviews",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=data_collator
)

# Train
trainer.train()

# Evaluate
eval_result = trainer.evaluate()
print("Perplexity:", math.exp(eval_result["eval_loss"]))

# After fine-tuning
print("\n=== AFTER FINE-TUNING ===\n")
for p in prompts:
    new_output = generate_text(p)
    print(f"Prompt: {p}")
    print(f"Before: {baseline[p][:120]}")
    print(f"After:  {new_output[:120]}\n")

# ====================== END ======================

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...


=== BEFORE FINE-TUNING ===

Prompt: This product is
Output: This product is designed to be used in conjunction with any other device and is intended for use in conjunction with any other electronic device.

The following devices may be inserted into any such device, even if the device does not have a user interface or an open device configuration:

a) any personal computer;

b) other personal computer or computer-based mobile device;

c) any computer-based home computer; and

d) any personal computer-based network-

Prompt: I bought this phone and
Output: I bought this phone and started working on it. I'm quite pleased with the way it performs, but this phone has the same design as the old Nexus 5. It's quite a bit thicker, and has a slightly better sound quality on the front touch panel than you'd expect from the Nexus 5. The other thing I like about the Nexus 5 is that it's a much smaller phone, which is nice, because it's smaller and there are no batteries in the back. I bought thi

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.890545
20,2.576917


Perplexity: 23.610351254493768

=== AFTER FINE-TUNING ===

Prompt: This product is
Before: This product is designed to be used in conjunction with any other device and is intended for use in conjunction with any
After:  This product is completely independent from pharmaceutical manufacturers and is tested on all your eyes. This product is

Prompt: I bought this phone and
Before: I bought this phone and started working on it. I'm quite pleased with the way it performs, but this phone has the same d
After:  I bought this phone and it really does a great job with my phone. And then when I read the reviews on the forums it was 

Prompt: The quality of this item
Before: The quality of this item is the same as that of any other item on the item page.
After:  The quality of this item was excellent!


Rated 1 out of 5 by Anonymous from Product does not hold up well In my bag I h

